# Training queue at learning rate 2e-4

Trains and scores the rank and alpha grid at learning rate 2e-4, with CLAHE on and off. Everything else matches the earlier clean-split runs: 4-bit base, split 2,786/357/361, seeds 42/3407, 43/3408 and 44/3409, checkpoint chosen by validation loss.

- **Phase A**, CLAHE on, 12 runs: rank/alpha 8/16, 32/16, 8/8, 32/32. Rank 16, alpha 16 is already in `final_01` and is linked in the summary, not rerun.
- **Phase B**, CLAHE off, 15 runs: rank/alpha 8/16, 16/16, 32/16, 8/8, 32/32.
- **Phase C**, timing only: re-scores seven earlier adapters that have no warm per-image timing. Off by default.

Each training run is followed by a Messidor score (archived 1,057 images) and a timing pass on the 361 test images. If the FP16 score overflows, a labelled FP32 fallback runs instead.

Steps: run **Setup** and **Status** (both only read files), then the **Pilot**, then the **Full queue**. Results go to `D:\Afterimage_Experiments\final_runs\final_02`.

## 1. Setup

In [1]:
from pathlib import Path
import importlib
import sys

if Path(sys.prefix).name.casefold() != "afterimage_env" or sys.version_info[:2] != (3, 11):
    raise RuntimeError(f"Select the Python 3.11 afterimage_env kernel. Current: {sys.executable}")

sys.dont_write_bytecode = True
QWEN_DIR = Path(r"D:\Afterimage_Reproduction\finetuning\Qwen3")
HELPER_DIR = QWEN_DIR / "paper_revision"  # pinned paper_reproduction.py stays in QWEN_DIR
for directory in (QWEN_DIR, HELPER_DIR):
    if str(directory) in sys.path:
        sys.path.remove(str(directory))
    sys.path.insert(0, str(directory))
importlib.invalidate_caches()

def load_helper(name):
    module = importlib.reload(sys.modules[name]) if name in sys.modules else importlib.import_module(name)
    if Path(module.__file__).resolve() != (HELPER_DIR / f"{name}.py").resolve():
        raise ImportError(f"Expected {name} from {HELPER_DIR}, got {module.__file__}")
    return module

final = load_helper("final_runs")
orch = load_helper("orchestrator_02")

# Switches
RUN_PHASE_A = True   # CLAHE on, 12 runs (runs first)
RUN_PHASE_B = True   # CLAHE off, 15 runs
RUN_PHASE_C = False  # timing only for seven earlier adapters, no training

SETTINGS = orch.default_settings()
SETTINGS["tag"] = "final_02"        # results folder; keep it to resume
SETTINGS["pilot_tag"] = "pilot_01"  # the pilot writes to final_02__pilot_01; change it to repeat the pilot
SETTINGS["phases"].update(A=RUN_PHASE_A, B=RUN_PHASE_B, C=RUN_PHASE_C)

# Messidor for CLAHE-off adapters: "archived_off" (same images from the archived export without CLAHE),
# "archived_on" (the CLAHE export) or None (skip).
SETTINGS["messidor_off"] = "archived_off"

print("Setup OK. Results folder:", orch.root(SETTINGS))
print("Stop file:", orch.stop_file(SETTINGS))

Setup OK. Results folder: D:\Afterimage_Experiments\final_runs\final_02
Stop file: D:\Afterimage_Experiments\final_runs\final_02\STOP


### Stopping after the current job

The queue checks for a file named `STOP` in the results folder before each job. To stop cleanly, create it while the queue runs, for example in PowerShell:

```
New-Item D:\Afterimage_Experiments\final_runs\final_02\STOP
```

The current job finishes, its logs are written, and the queue stops. The notebook kernel is busy while the queue runs, so the file has to come from outside (Explorer or a terminal). Starting the queue again removes an old `STOP` file. Interrupting the kernel also works, but it kills the running job; that job resumes from its last saved epoch next time.

## 2. Status

Lists every job as done, pending, failed or running, with an estimated time per job. Starts nothing.

In [2]:
jobs = orch.show_status(SETTINGS)

Queue in D:\Afterimage_Experiments\final_runs\final_02  (learning rate 0.0002)

Phase A: CLAHE on
  [pending ] train r8_on_alpha16_lr0p0002_t42_a3407  (~69 min)
  [pending ] score r8_on_alpha16_lr0p0002_t42_a3407__messidor_archived  (~15 min)
  [pending ] score r8_on_alpha16_lr0p0002_t42_a3407__aptos361_on  (~16 min)
  [pending ] train r8_on_alpha16_lr0p0002_t43_a3408  (~69 min)
  [pending ] score r8_on_alpha16_lr0p0002_t43_a3408__messidor_archived  (~15 min)
  [pending ] score r8_on_alpha16_lr0p0002_t43_a3408__aptos361_on  (~16 min)
  [pending ] train r8_on_alpha16_lr0p0002_t44_a3409  (~69 min)
  [pending ] score r8_on_alpha16_lr0p0002_t44_a3409__messidor_archived  (~15 min)
  [pending ] score r8_on_alpha16_lr0p0002_t44_a3409__aptos361_on  (~16 min)
  [pending ] train r32_on_alpha16_lr0p0002_t42_a3407  (~69 min)
  [pending ] score r32_on_alpha16_lr0p0002_t42_a3407__messidor_archived  (~15 min)
  [pending ] score r32_on_alpha16_lr0p0002_t42_a3407__aptos361_on  (~16 min)
  [pending ] tr

## 3. Pilot

One variant from each enabled phase: 10 training steps, then Messidor and timing on 12 images each. It writes to the pilot folder, not `final_02`, and takes about 30 minutes. Start the full queue only after it prints **Pilot passed**.

In [3]:
orch.show_status(SETTINGS, pilot=True)
orch.run_pilot(SETTINGS)

Pilot in D:\Afterimage_Experiments\final_runs\final_02__pilot_01  (learning rate 0.0002)

Phase A: CLAHE on
  [done    ] train r8_on_alpha16_lr0p0002_t42_a3407  (~6 min)
  [done    ] score r8_on_alpha16_lr0p0002_t42_a3407__messidor_archived  (~4 min)
  [done    ] score r8_on_alpha16_lr0p0002_t42_a3407__aptos361_on  (~4 min)

Phase B: CLAHE off
  [done    ] train r8_off_alpha16_lr0p0002_t42_a3407  (~6 min)
  [done    ] score r8_off_alpha16_lr0p0002_t42_a3407__messidor_archived_off  (~4 min)
  [done    ] score r8_off_alpha16_lr0p0002_t42_a3407__aptos361_off  (~4 min)

6 jobs (2 training runs): 6 done. Estimated GPU time left: about 0.0 h (from logged durations of earlier jobs).
Free space on D:\: 423 GB.
Idle baseline: 36.0 W mean over 58 samples.

Queue finished.
  done earlier                 r8_on_alpha16_lr0p0002_t42_a3407
  done earlier                 r8_on_alpha16_lr0p0002_t42_a3407__messidor_archived
  done earlier                 r8_on_alpha16_lr0p0002_t42_a3407__aptos361_on
  d

{'r8_on_alpha16_lr0p0002_t42_a3407': 'done earlier',
 'r8_on_alpha16_lr0p0002_t42_a3407__messidor_archived': 'done earlier',
 'r8_on_alpha16_lr0p0002_t42_a3407__aptos361_on': 'done earlier',
 'r8_off_alpha16_lr0p0002_t42_a3407': 'done earlier',
 'r8_off_alpha16_lr0p0002_t42_a3407__messidor_archived_off': 'done earlier',
 'r8_off_alpha16_lr0p0002_t42_a3407__aptos361_off': 'done earlier'}

## 4. Full queue

Starts or resumes the queue: phase A, then B, then C, each config's three seeds before the next config. Finished jobs are skipped; an interrupted training run resumes from its last saved epoch. A failed job is recorded and the queue moves on. It keeps the PC awake and uses the shared GPU lock, so it will not start while another GPU job holds it.

In [4]:
outcomes = orch.run_queue(SETTINGS)

Idle baseline: 29.5 W mean over 58 samples.

[1/81] Phase A: CLAHE on: r8_on_alpha16_lr0p0002_t42_a3407
Log: D:\Afterimage_Experiments\final_runs\final_02\logs\r8_on_alpha16_lr0p0002_t42_a3407\attempts\20261122T071426Z-oatk71s3\launch.log
Prepared {'train': 2786, 'val': 357, 'test': 361} APTOS images in D:\Afterimage_Experiments\final_runs\final_02\train\r8_on_alpha16_lr0p0002_t42_a3407
Variant r8_on_alpha16_lr0p0002_t42_a3407: overrides {'lr': 0.0002, 'lora_alpha': 16}
Starting train. Log: D:\Afterimage_Experiments\final_runs\final_02\train\r8_on_alpha16_lr0p0002_t42_a3407\attempts\20261122T071436Z-7ruto0x1\train.log
train: 30 s elapsed; utilization logged in 20261122T071436Z-7ruto0x1.
train: 61 s elapsed; utilization logged in 20261122T071436Z-7ruto0x1.
train: 91 s elapsed; utilization logged in 20261122T071436Z-7ruto0x1.
train: 121 s elapsed; utilization logged in 20261122T071436Z-7ruto0x1.
train: 151 s elapsed; utilization logged in 20261122T071436Z-7ruto0x1.
train: 182 s elapsed; 

In [ ]:
outcomes = orch.run_queue(SETTINGS)

## 5. Summary

Writes `summary.md` and `summary.csv` in the results folder: accuracy, QWK, macro-F1, AUC, collapsed flag, scoring precision, timing and energy per run, and the full 2e-4 grid including the three rank 16 runs from `final_01`. Reads saved files only.

In [5]:
orch.energy_per_job(orch.root(SETTINGS))
rows = orch.summarize(SETTINGS)
summary = orch.root(SETTINGS) / "summary.md"
if summary.is_file():
    from IPython.display import Markdown, display
    display(Markdown(summary.read_text(encoding="utf-8")))

Summary of 30 row(s): D:\Afterimage_Experiments\final_runs\final_02\summary.md


# Training queue summary (final_02)

Learning rate 0.0002, 4-bit base, split 2,786/357/361, checkpoint chosen by validation loss. Accuracy, QWK, macro-F1 and macro AUC come from the pinned five-answer scorer on the 361 clean test images. Collapsed means every test image got the same grade. Timing excludes model loading and the first three images. Energy is whole-card GPU power (see energy_note.txt).

## Grid at this learning rate

Rank 16, alpha 16, CLAHE on comes from final_01 (not rerun).

| CLAHE | Rank | Alpha | Source | Seeds | Scoring | Accuracy % | QWK | Macro-F1 | Macro AUC | Collapsed | Messidor accuracy % | Messidor QWK | Median s/image | Train h | Energy Wh (all jobs) |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| on | 8 | 8 | final_02 | 1 | FP16 | 78.39 | 0.8917 | 0.6430 | 0.9446 | 0 |  |  |  | 1.55 | 438 |
| on | 8 | 16 | final_02 | 3 | FP16 | 78.21 ± 1.05 | 0.8958 ± 0.0011 | 0.6372 | 0.9370 | 0 | 50.93 | 0.6388 | 0.792 | 1.71 | 575 |
| on | 16 | 16 | final_01 | 3 | FP16 | 78.12 ± 2.22 | 0.8939 ± 0.0069 | 0.6291 | 0.9415 | 0 | 48.53 | 0.6029 |  | 1.39 |  |
| on | 32 | 16 | final_02 | 3 | FP16 | 74.52 ± 2.64 | 0.8866 ± 0.0229 | 0.6049 | 0.9394 | 0 | 50.93 | 0.6298 | 0.815 | 1.05 | 389 |

## Runs

| Phase | Run | Status | Scoring | Accuracy | QWK | Macro-F1 | AUC | Collapsed | Messidor acc | Median s/image | Train h | Train img/s | Energy Wh |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| A | r8_on_alpha16_lr0p0002_t42_a3407 | scored | FP16 | 77.01% | 0.8969 | 0.6418 | 0.9473 |  | 50.43% | 0.809 | 1.55 | 3.01 | 530 |
| A | r8_on_alpha16_lr0p0002_t43_a3408 | scored | FP16 | 78.95% | 0.8947 | 0.6197 | 0.9340 |  | 51.84% | 0.808 | 1.55 | 3.01 | 530 |
| A | r8_on_alpha16_lr0p0002_t44_a3409 | scored | FP16 | 78.67% | 0.8958 | 0.6500 | 0.9298 |  | 50.52% | 0.758 | 2.03 | 3.07 | 664 |
| A | r32_on_alpha16_lr0p0002_t42_a3407 | scored | FP16 | 74.79% | 0.8993 | 0.6208 | 0.9399 |  | 51.66% | 0.889 | 1.06 | 2.95 | 392 |
| A | r32_on_alpha16_lr0p0002_t43_a3408 | scored | FP16 | 71.75% | 0.8601 | 0.5693 | 0.9343 |  | 50.80% | 0.788 | 1.07 | 2.92 | 390 |
| A | r32_on_alpha16_lr0p0002_t44_a3409 | scored | FP16 | 77.01% | 0.9004 | 0.6247 | 0.9440 |  | 50.33% | 0.769 | 1.01 | 3.09 | 384 |
| A | r8_on_alpha8_lr0p0002_t42_a3407 | scored | FP16 | 78.39% | 0.8917 | 0.6430 | 0.9446 |  |  |  | 1.55 | 3.03 | 438 |
| A | r8_on_alpha8_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| A | r8_on_alpha8_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| A | r32_on_alpha32_lr0p0002_t42_a3407 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| A | r32_on_alpha32_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| A | r32_on_alpha32_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r8_off_alpha16_lr0p0002_t42_a3407 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r8_off_alpha16_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r8_off_alpha16_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r16_off_alpha16_lr0p0002_t42_a3407 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r16_off_alpha16_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r16_off_alpha16_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r32_off_alpha16_lr0p0002_t42_a3407 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r32_off_alpha16_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r32_off_alpha16_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r8_off_alpha8_lr0p0002_t42_a3407 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r8_off_alpha8_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r8_off_alpha8_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r32_off_alpha32_lr0p0002_t42_a3407 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r32_off_alpha32_lr0p0002_t43_a3408 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| B | r32_off_alpha32_lr0p0002_t44_a3409 | not trained |  |  |  |  |  |  |  |  |  |  |  |
| linked | r16_on_lr0p0002_t42_a3407 | scored | FP16 | 75.90% | 0.8890 | 0.6241 | 0.9436 |  | 49.57% |  | 1.45 | 3.22 |  |
| linked | r16_on_lr0p0002_t43_a3408 | scored | FP16 | 80.33% | 0.9018 | 0.6567 | 0.9439 |  | 49.95% |  | 1.59 | 2.95 |  |
| linked | r16_on_lr0p0002_t44_a3409 | scored | FP16 | 78.12% | 0.8909 | 0.6064 | 0.9370 |  | 46.07% |  | 1.13 | 2.77 |  |

Messidor for CLAHE-off adapters: the same 1,057 archived images, read from the export without CLAHE (cohorts/messidor_archived_off.csv).


## 6. What is logged

Every attempt of every job has its own folder, `logs\<job>\attempts\<time>`, with:

- `command.json`: the exact command and environment
- `system_specs.json`: GPU, driver, CUDA, CPU, RAM, and torch, transformers, unsloth and peft versions
- the stage log (`launch.log`, plus copies of the runner's `train.log` and `score.log` for training; `score.log` for scoring)
- `utilization.csv`: GPU %, VRAM, CPU % and RAM every 5 s
- `run_report.json`: status, wall time, peak VRAM, peak PyTorch memory, training images per second

`logs\<job>\metrics.json` holds the metrics. Per-image predictions with probabilities are in `train\<run>\aptos_test.predictions.csv` and `scores\<job>\predictions.csv`.

In the results folder: `power_log.csv` (GPU power every 5 s), `power_events.csv` (job start and end), `idle_baseline.json` (60 s of idle power at each queue start), `energy_per_job.csv` and `queue_state.json`. Power is for the whole graphics card, including the display; the CPU and the rest of the PC are not measured.

Before each training run the queue warns if D: has less than 30 GB free.